# Field Shift: export data for the season replay

Run top to bottom in Colab (about 5 minutes). At the end your browser downloads two files:

| File | What |
|---|---|
| `power_daily_replay.csv` | NASA POWER daily weather 1984-2025 at the demo field (max/min temp, rain, humidity, wind, sunlight) |
| `cmip6_monthly.csv` | NASA NEX-GDDP-CMIP6 monthly means per model: 1985-2014 and 2030-2050, two scenarios |

Copies are also saved in `field_shift_exports` on Drive. **Attach both to the chat**, plus `fields_soil.csv` and
`fields_terrain.csv` from round 2 if you have them.

## 1. Settings (same field as the other exports)

In [ ]:
PROJECT_ID = "green-line-514704"
DRIVE_FOLDER = "field_shift_exports"
DEMO_NAME = "ellis_county_tx"
LAT, LON = 32.25, -96.75
FIRST_YEAR, LAST_YEAR = 1984, 2025

SCENARIOS = ["ssp245", "ssp585"]
HIST_PERIOD = ("1985-01-01", "2015-01-01")
FUTURE_PERIOD = ("2030-01-01", "2051-01-01")
MODELS = ["ACCESS-CM2", "CanESM5", "EC-Earth3", "GFDL-ESM4",
          "INM-CM5-0", "MIROC6", "MPI-ESM1-2-HR", "MRI-ESM2-0"]

## 2. Connect Drive and Earth Engine

In [ ]:
import os, time, shutil, requests
import numpy as np, pandas as pd
from google.colab import drive, files
drive.mount("/content/drive")
DRIVE_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs(DRIVE_DIR, exist_ok=True)

import ee
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
print("Ready")

## 3. Start the climate-projection job in Earth Engine (runs while step 4 downloads)

In [ ]:
pt = ee.Geometry.Point([LON, LAT])
gddp = ee.ImageCollection("NASA/GDDP-CMIP6")

available = set(gddp.filter(ee.Filter.eq("scenario", SCENARIOS[0]))
                    .filterDate("2040-01-01", "2040-01-03")
                    .aggregate_array("model").distinct().getInfo())
models = [m for m in MODELS if m in available]
print("Models used:", models, "| not in catalog:", sorted(set(MODELS) - available) or "none")

def monthly_means(model, scenario, start, end):
    col = (gddp.filter(ee.Filter.eq("model", model))
               .filter(ee.Filter.eq("scenario", scenario))
               .filterDate(start, end)
               .select(["tasmax", "tasmin", "pr"]))
    def per_month(m):
        m = ee.Number(m)
        img = col.filter(ee.Filter.calendarRange(m, m, "month")).mean()
        vals = img.reduceRegion(ee.Reducer.mean(), pt, 27830)
        return ee.Feature(None, vals).set({"model": model, "scenario": scenario, "month": m})
    return ee.FeatureCollection(ee.List.sequence(1, 12).map(per_month))

parts = []
for model in models:
    parts.append(monthly_means(model, "historical", *HIST_PERIOD))
    for scen in SCENARIOS:
        parts.append(monthly_means(model, scen, *FUTURE_PERIOD))

cmip_task = ee.batch.Export.table.toDrive(
    collection=ee.FeatureCollection(parts).flatten(),
    description=f"{DEMO_NAME}_cmip6_monthly", folder=DRIVE_FOLDER,
    fileNamePrefix="cmip6_monthly", fileFormat="CSV",
    selectors=["model", "scenario", "month", "tasmax", "tasmin", "pr"])
cmip_task.start()
print("Started. Continue with step 4 while it runs.")

## 4. Download NASA POWER daily weather

In [ ]:
POWER_CSV = f"{DRIVE_DIR}/power_daily_replay.csv"
PARAMS = {"T2M_MAX": "tmax", "T2M_MIN": "tmin", "PRECTOTCORR": "rain",
          "RH2M": "rh", "WS2M": "wind", "ALLSKY_SFC_SW_DWN": "srad"}

def power_chunk(y0, y1):
    url = ("https://power.larc.nasa.gov/api/temporal/daily/point"
           f"?parameters={','.join(PARAMS)}&community=AG&longitude={LON}&latitude={LAT}"
           f"&start={y0}0101&end={y1}1231&format=JSON")
    for attempt in range(4):
        try:
            r = requests.get(url, timeout=300)
            r.raise_for_status()
            p = r.json()["properties"]["parameter"]
            df = pd.DataFrame(p)
            df.index = pd.to_datetime(df.index, format="%Y%m%d")
            return df
        except Exception as e:
            print(f"  {y0}-{y1} attempt {attempt + 1} failed: {e}")
            time.sleep(10 * (attempt + 1))
    raise RuntimeError(f"POWER download failed for {y0}-{y1}")

if os.path.exists(POWER_CSV):
    raw = pd.read_csv(POWER_CSV, index_col=0, parse_dates=True)
    print("Loaded saved POWER file")
else:
    chunks = []
    for y0 in range(FIRST_YEAR, LAST_YEAR + 1, 10):
        y1 = min(y0 + 9, LAST_YEAR)
        print("Downloading", y0, "-", y1)
        chunks.append(power_chunk(y0, y1))
    raw = pd.concat(chunks).rename(columns=PARAMS)
    raw.to_csv(POWER_CSV)


raw = raw.replace(-999, np.nan)
print(raw.describe().round(1).loc[["count", "mean", "min", "max"]])

## 5. Wait for Earth Engine, then download both files

In [ ]:
while cmip_task.status()["state"] in ("READY", "RUNNING"):
    print("Earth Engine still running...", cmip_task.status()["state"]); time.sleep(30)
st = cmip_task.status()
if st["state"] != "COMPLETED":
    raise RuntimeError(f"Earth Engine export failed: {st.get('error_message')}")
CMIP_CSV = f"{DRIVE_DIR}/cmip6_monthly.csv"
for _ in range(40):                      # Drive can take a minute to show the file
    if os.path.exists(CMIP_CSV):
        break
    time.sleep(15)
cmip = pd.read_csv(CMIP_CSV)
print(cmip.groupby(["scenario"]).model.nunique().rename("models"))
print("Rows with missing values:", int(cmip.isna().any(axis=1).sum()))

shutil.copy(POWER_CSV, "/content/power_daily_replay.csv")
shutil.copy(CMIP_CSV, "/content/cmip6_monthly.csv")
files.download("/content/power_daily_replay.csv")
files.download("/content/cmip6_monthly.csv")
print("Downloaded. Attach both files to the chat.")